# MANATUABON PHASE 9: KAGGLE LORA FINE-TUNING

## NVIDIA Nemotron-3-Nano-30B Reasoning Optimization
Run this notebook on **Google Colab Pro** with an **A100 or L4 GPU**.
Make sure to upload `synthetic_cot.jsonl` (generated by `kaggle_pipeline.py`) to the Colab workspace before running.

In [1]:
# 1. Install Required Dependencies (Including Mamba-SSM for Hybrid Architecture)
!pip install -U transformers peft accelerate bitsandbytes datasets trl mamba-ssm causal-conv1d

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.7/121.7 kB 12.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 165.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.8/526.8 kB 74.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 63.1 MB/s eta 0:00:00
Using cached ninja-1.13.0-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (180 kB)
  Created wheel for mamba-ssm: filename=mamba_ssm-2.3.1-

In [2]:
import os
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, EarlyStoppingCallback
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

def format_prompt(sample):
    """
    Format the ShareGPT conversational format generated by kaggle_pipeline.py
    into a continuous prompt string for Next-Token Prediction.
    """
    convo = sample["conversations"]
    prompt = convo[0]["value"]
    response = convo[1]["value"]

    # Simple template: User prompt + Assistant reasoning and \boxed{} answer
    text = f"User: {prompt}\n\nAssistant: {response}"
    return {"text": text}

# 2. Load the generated dataset (Upload synthetic_cot.jsonl to your Colab workspace)
dataset_path = "synthetic_cot.jsonl"
if not os.path.exists(dataset_path):
    print(f"\n❌ ERROR: Please upload {dataset_path} to your Colab workspace!")
else:
    dataset = load_dataset("json", data_files=dataset_path, split="train")
    dataset = dataset.map(format_prompt)

    # Create a 90/10 Train/Eval split for Early Stopping
    dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
    train_data = dataset_split["train"]
    eval_data = dataset_split["test"]
    print(f"\n✅ Loaded {len(train_data)} training and {len(eval_data)} validation traces.")

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/95 [00:00<?, ? examples/s]


✅ Loaded 85 training and 10 validation traces.


In [4]:
# 3. Configure 8-bit Quantization (more stable for hybrid Mamba+MoE models)
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
)

# 4. Load the Nemotron 30B Nano Model & Tokenizer
model_id = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
print(f"Loading base model: {model_id}...")

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = 4096 # Bypass strict TRL parsing blocks

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16, # Force bfloat16 to match training config and MoE routing
    trust_remote_code=True,
)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()


Loading base model: nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16...


config.json: 0.00B [00:00, ?B/s]

configuration_nemotron_h.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16:
- configuration_nemotron_h.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer_config.json: 0.00B [00:00, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16:
- modeling_nemotron_h.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/197 [00:00<?, ?B/s]

In [7]:
# HOTFIX: Patch NVIDIA Nemotron MoE dtype mismatch for LoRA training
import glob, os

pattern = os.path.expanduser(
    "~/.cache/huggingface/modules/transformers_modules/nvidia/**/modeling_nemotron_h.py"
)
files = glob.glob(pattern, recursive=True)

for f in files:
    with open(f, 'r') as fh:
        content = fh.read()

    old = "final_hidden_states.index_add_(0, token_indices, weighted_output)"
    new = "final_hidden_states.index_add_(0, token_indices, weighted_output.to(final_hidden_states.dtype))"

    if old in content and new not in content:
        content = content.replace(old, new)
        with open(f, 'w') as fh:
            fh.write(content)
        print(f"✅ Patched MoE dtype bug in: {f}")
    else:
        print(f"ℹ️ Already patched or not found in: {f}")

if not files:
    print("⚠️ No cached file found yet. Run Cell 3 first, then come back here.")


✅ Patched MoE dtype bug in: /root/.cache/huggingface/modules/transformers_modules/nvidia/NVIDIA_hyphen_Nemotron_hyphen_3_hyphen_Nano_hyphen_30B_hyphen_A3B_hyphen_BF16/cbd3fa9f933d55ef16a84236559f4ee2a0526848/modeling_nemotron_h.py


In [10]:
# 5. Configure the LoRA Adapter (Rank 32 maximum per Kaggle rules)
print("Configuring Rank-32 LoRA adapter...")
peft_config = LoraConfig(
    r=32,
    lora_alpha=64,
    target_modules="all-linear",
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

# Force all trainable LoRA weights to BFloat16 to match Nemotron MoE hidden states
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.bfloat16)

model.print_trainable_parameters()


Configuring Rank-32 LoRA adapter...


/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 888,154,112 || all params: 32,466,091,456 || trainable%: 2.7356


In [12]:
# 6. Set up the SFTTrainer with Early Stopping
output_dir = "./nemotron_reasoning_lora"
sft_config = SFTConfig(
    output_dir=output_dir,
    dataset_text_field="text",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=150,
    learning_rate=2e-4,
    bf16=True,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="steps",
    save_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    optim="paged_adamw_8bit",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=eval_data,
    processing_class=tokenizer,
    args=sft_config,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# 7. Train the model
print("\n🔥 Starting LoRA Fine-tuning with Early Stopping...")
trainer.train()


🔥 Starting LoRA Fine-tuning with Early Stopping...


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


RuntimeError: index_add_(): self (BFloat16) and source (Float) must have the same scalar type

In [ ]:
# 8. Save the final adapter for Kaggle submission
print(f"\n✅ Training complete. Saving LoRA adapter to {output_dir}")
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print("Adapter is ready to be zipped and submitted to Kaggle!")